# MolPLAtte — pocket-conditioned lead optimization

Load one checkpoint and its artifacts, then optimize **any** flavor compound against
**any** protein structure.

The query is built from three things, none of which is derived from the answer:

| input | what it is |
|---|---|
| **core template** | the molecule you want to modify, minus one substituent |
| **flavor profile** | the sensory profile you are aiming for (24 bits) |
| **pocket** | the receptor you want it to bind (ESM-2 embedding of the 10 Å site) |

That last property is the point. The project's original condition vector was an RDKit
fragment vector computed from the *intact* molecule — so it contained the R-group being
retrieved. Zeroing it at inference took hit@1 from 0.6111 to exactly 0.0000. Everything
here is exogenous by construction.

## 1. Setup

Point these at your checkpoint and library. The `condvec_dim` / `pocket_input_dim` must match how the checkpoint was **trained** — the query projector's input width is a function of both, so a mismatch raises at load rather than producing a quietly wrong model.

In [ ]:
import sys, json
from pathlib import Path

# Walk up looking for the package rather than assuming a fixed depth, so the
# notebook runs from the repo, from a copy, or from a papermill working dir.
SRC = next((p / "src" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "lead_optimization.py").is_file()), None)
if SRC is None:
    raise RuntimeError(
        "cannot find lead_optimization.py -- run this notebook from inside the "
        "MolPLAtte repo, or set SRC to <repo>/molplatte/src by hand")
sys.path.insert(0, str(SRC))
print("src:", SRC)

import numpy as np
import torch

from lead_optimization import LeadOptimizer, pocket_embedding_from_structure

DATA       = Path.home() / "preprocessed" / "molplatte"
CHECKPOINT = Path.home() / "checkpoints" / "molplatte" / "molplatte-final-s911012.pt"
VOCAB      = DATA / "union_vocab" / "base-full__crossdocked__tastepocket" / "rgroup_vocab.pkl.gz"

# Must match how the checkpoint was TRAINED.
#   flavour-only      : condvec_dim=24,   pocket_input_dim=0
#   pocket-conditioned: condvec_dim=1304, pocket_input_dim=1280
# Add assembly_head="AssemblyHead" only for a checkpoint trained with
# assembly.enabled=true; on any other checkpoint the head loads at random init
# and `opt.has_assembly_head` reports False rather than emitting nonsense.
# The shipped checkpoint is pocket-CAPABLE: it accepts a 1280-d pocket vector.
# The pocket reduction is UNTRAINED (zero-init output), so a supplied pocket
# contributes exactly zero -- deliberate, not an oversight. Pocket conditioning
# measured null at 0, 1,056 and 168,096 trainable parameters; see
# docs/step3_pocket_capacity_2026-09-09.md.
MODEL_KWARGS = dict(condvec_dim=1304, pocket_input_dim=1280, pocket_dim=32,
                    assembly_head="AssemblyHead")   # the checkpoint carries one

print("checkpoint:", CHECKPOINT.name, "exists:", CHECKPOINT.exists())
print("library   :", VOCAB.name, "exists:", VOCAB.exists())

## 2. Load the model and embed the library

`build_library()` embeds every R-group with the **current** weights. Both projectors are trained, so an index built by a different checkpoint scores against a different space and returns plausible nonsense — always rebuild after changing checkpoints.

In [ ]:
opt = LeadOptimizer.load(CHECKPOINT, VOCAB, device="cuda", **MODEL_KWARGS)
opt.build_library(batch_size=1024)

print(f"library rows      {len(opt.vocab):,}")
print(f"effective size    {opt.vocab.effective_size():.0f}")
print(f"top-1 share       {100 * opt.vocab.frequency_prior[0]:.2f}%")
print(f"novel rows        {len(opt._novel):,}  (absent from the pretraining vocabulary)")

`effective size` is `exp(H)` of the R-group frequency distribution — how many-way the
retrieval *actually* is, as opposed to the nominal row count. A library of 92,000 rows
with an effective size near 950 means Hit@K should be read against the frequency prior,
not against 1/92000.

## 3. Flavor-only optimization

Start without a pocket. Every suggestion is scored with the logQ popularity correction: InfoNCE optimizes PMI, `log p(k|q) − log p(k)`, so `log p(k)` has to be added back before the ranking is comparable to a frequency prior.

In [ ]:
VANILLIN = "COc1cc(C=O)ccc1O"

results = opt.optimize(VANILLIN, flavor=["sweet", "woody"], top_k=8)

for slot in results:
    print(f"\ndecomposition {slot.decomp_index}, slot {slot.slot_index}"
          f"   original substituent: {slot.original_rgroup}")
    for s in slot.suggestions:
        flag = "  [novel]" if s.is_novel else ""
        print(f"   #{s.rank:<2} {s.smiles:<28} score {s.score:+.4f}"
              f"   corpus count {s.corpus_count:>7,}{flag}")

### Reading the output

- **`original substituent`** is what was removed. The model does not see it.
- **`corpus count`** is how often that R-group occurs. A high-count suggestion at rank 1
  may be the prior talking rather than the model — compare against the counts of the
  lower ranks.
- **`[novel]`** marks an R-group absent from the pretraining vocabulary, i.e. one that
  entered through CrossDocked or tastepocket. These are the interesting hits: they show
  retrieval reaching past the chemistry the model was pretrained on.

## 4. Steering with the flavor profile

The same core, asked for different sensory outcomes. If the two rankings are identical, the condition is not doing any work — which is exactly what the shuffle test measures at training time.

In [ ]:
def top_smiles(smiles, flavor, k=6):
    res = opt.optimize(smiles, flavor=flavor, top_k=k)
    return [s.smiles for s in res[0].suggestions] if res else []

a = top_smiles(VANILLIN, ["sweet"])
b = top_smiles(VANILLIN, ["bitter", "medicinal"])

print(f"{'sweet':<32}{'bitter + medicinal'}")
for x, y in zip(a, b):
    print(f"{x:<32}{y}")

# Compare ORDER, not set membership. Two conditions can return the same six
# R-groups in different order, which IS the condition steering retrieval; a
# set-overlap test calls that "no signal" and is why this cell read as a null
# even after the scoring bug below was fixed.
overlap = len(set(a) & set(b))
print(f"\nsame set in the top {len(a)}: {overlap}/{len(a)}")
print(f"same ORDER: {a == b}")
print("identical order => the flavor condition is carrying no signal here"
      if a == b else "order differs => the condition is steering retrieval")

## 5. Adding a pocket

`pocket_embedding_from_structure` takes a `.pdb` or `.cif`, finds the ligand, selects
protein residues within 10 Å, and mean-pools ESM-2 embeddings over them.

Three details are load-bearing and are why you should not roll your own:

- **10 Å** matches the `pocket10` convention the corpus uses. A different cutoff puts the
  query in a different space from the library.
- **Pooling is over every pocket residue of every chain at once**, not per-chain then
  averaged — otherwise a 5-residue contact counts as much as a 40-residue wall.
- **ESM prepends a BOS token**, so residue *i* is token *i+1*. The offset is verified
  against a probe rather than assumed; getting it wrong shifts every pocket by one
  residue and raises nothing.

Verified against a stored corpus embedding: cosine 1.000000.

In [ ]:
# Requires a checkpoint trained WITH the pocket half:
#   MODEL_KWARGS = dict(condvec_dim=1304, pocket_input_dim=1280, pocket_dim=32,
#                       assembly_head="AssemblyHead")
STRUCTURE = Path.home() / "datasets/tastepocket/structures/cif/8F76.cif"   # OR51E2 + propionate

if opt.model.config.pocket_input_dim == 0:
    print("This checkpoint is flavor-only; reload with the pocket kwargs to run this cell.")
elif not STRUCTURE.exists():
    print(f"no structure at {STRUCTURE}")
else:
    pocket = pocket_embedding_from_structure(STRUCTURE, device="cuda")
    print(f"pocket embedding {pocket.shape}, norm {np.linalg.norm(pocket):.3f}")

    with_pocket = opt.optimize(VANILLIN, flavor=["sweet"], pocket=pocket, top_k=6)
    without     = opt.optimize(VANILLIN, flavor=["sweet"], pocket=None,   top_k=6)

    print(f"\n{'with pocket':<32}{'flavor only'}")
    for p, q in zip(with_pocket[0].suggestions, without[0].suggestions):
        print(f"{p.smiles:<32}{q.smiles}")

Passing `pocket=None` is not the same as passing zeros by accident — it is the honest
encoding of "no pocket supplied". `PocketConditioning` masks an all-zero pocket half to
exactly zero, so the query falls back to flavor alone rather than to noise, and a
flavor-only run stays numerically identical to `condvec_dim=24`.

## 6. Building the actual molecule

Retrieval says *which* R-group belongs at a joint. It cannot say *how to attach
it*: the joint is masked, so the shared linker atom's identity and the bond that
reforms were both replaced by MASK sentinels. The **assembly head** predicts what
the mask destroyed, which is what turns a ranked list of fragments into
optimized molecules.

`assemble=True` needs a checkpoint trained with `assembly.enabled=true`. Without
one, every suggestion returns `product=None` and states why, rather than
silently coming back unassembled.

In [ ]:
if not opt.has_assembly_head:
    print("This checkpoint has no assembly head — retrain with assembly.enabled=true")
else:
    for slot in opt.optimize(VANILLIN, flavor=["sweet"], top_k=6, assemble=True)[:1]:
        print(f"removed: {slot.original_rgroup}\n")
        for s in slot.suggestions:
            if s.product:
                flag = "" if s.aromaticity_kept else "   <- aromaticity broken"
                print(f"  #{s.rank} {s.smiles:<16} -> {s.product}{flag}")
            else:
                print(f"  #{s.rank} {s.smiles:<16} -- {s.product_error}")

### Read `aromaticity_kept` before trusting a product

Every attribute at the joint — atom identity, formal charge, hydrogen count,
bond order — is predicted by an independent classifier, so they can disagree
with each other. RDKit will happily sanitize a molecule whose aromatic ring has
been broken by a wrong hydrogen count, so **"it sanitized" and "it is chemically
right" are different claims**.

Measured on a deliberately undertrained head: 100% sanitized, 71% preserved
aromaticity. The gap is entirely the head, not the assembly machinery — a
round-trip control that reattaches the *original* R-group using stored joint
chemistry is exact 6/6, and is guarded in the test suite.

## 7. Your own molecule and receptor

Edit and run.

In [ ]:
MY_SMILES    = "CC(=O)OC1=CC=CC=C1C(=O)O"       # any flavor compound
MY_FLAVOR    = ["bitter"]                        # see opt.flavor_vector for the vocabulary
MY_STRUCTURE = None                              # Path("/path/to/receptor.pdb")

pocket = None
if MY_STRUCTURE and opt.model.config.pocket_input_dim:
    pocket = pocket_embedding_from_structure(MY_STRUCTURE, device="cuda")

for slot in opt.optimize(MY_SMILES, flavor=MY_FLAVOR, pocket=pocket, top_k=10):
    print(f"\nslot {slot.slot_index} (decomposition {slot.decomp_index})"
          f"   original: {slot.original_rgroup}")
    for s in slot.suggestions:
        print(f"   {s}")

### The 24 flavor labels

`sweet, bitter, sour, salty, umami` (taste) · `fruity, green, floral, fatty, woody, spicy,
roasted, sulfurous, earthy, nutty, herbal, medicinal, citrus, dairy, alcoholic, meaty,
minty` (odor) · `odorless` · `unknown`

`odorless` and `unknown` are distinct claims: the first says nothing is smelled, the
second says nothing is known. An empty request becomes `unknown` rather than an all-zero
vector, because zeros would assert "no flavor at all" — a claim you did not make.